# 5주차 · 카라멜팝콘 SQL 집계와 검증

공용 Postgres의 수업용 가상데이터를 사용합니다. 파일 → Drive에 사본 저장 후 위에서 아래로 실행하세요. 비밀번호는 수업 중 안내합니다.

[설치·접속 및 과제 안내](https://github.com/team-datapopcorn/bdai13-data-handling-weekly/tree/master/week05_sql_aggregation)

## 1. 준비와 접속
접속 후 주문 5행이 보이면 성공입니다.

In [ ]:
!pip install psycopg2-binary -q

In [ ]:
from getpass import getpass
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="aws-1-us-east-1.pooler.supabase.com",
    port=6543,
    dbname="postgres",
    user="bdai13_student.ihenzvyxmlqmfrwjeknd",
    password=getpass("수업에서 안내한 비밀번호: "),
    sslmode="require",
    connect_timeout=15,
)
conn.autocommit = True

def query(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[col[0] for col in cur.description])

query("SELECT * FROM bdai13.orders LIMIT 5")

## 2. 테이블 구조 확인
컬럼과 연결 키를 확인하고 고객·주문·주문항목·상품의 관계를 적어보세요.

In [ ]:
query("""
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'bdai13'
  AND table_name IN ('customers', 'products', 'orders', 'order_items', 'logins')
ORDER BY table_name, ordinal_position
""")

## 3. 원본으로 기준 건수 확인
이 원본을 pandas 검증에도 사용합니다.

In [ ]:
customers = query("SELECT * FROM bdai13.customers")
products = query("SELECT * FROM bdai13.products")
orders = query("SELECT * FROM bdai13.orders")
order_items = query("SELECT * FROM bdai13.order_items")
pd.DataFrame({"table": ["customers", "products", "orders", "order_items"],
              "rows": [len(customers), len(products), len(orders), len(order_items)]})

## 4. AI에게 집계 쿼리 3개 요청
지역별 총 매출·주문 건수, 카테고리별 판매 수량 상위 3개, 월별 매출 추이를 구하세요. 컬럼 정보를 함께 제공하고, 매출 정의와 NULL 처리·동률 처리 기준을 정하세요. AI가 만든 SELECT를 아래 셀에 넣습니다.

### 지역별 총 매출·주문 건수

In [ ]:
sql = """
-- AI가 만든 SELECT 쿼리를 여기에 넣으세요.
"""
# 쿼리를 넣은 뒤 아래 줄의 주석을 해제하세요.
# result = query(sql)
# display(result)

### 카테고리별 판매 수량 상위 3개

In [ ]:
sql = """
-- AI가 만든 SELECT 쿼리를 여기에 넣으세요.
"""
# 쿼리를 넣은 뒤 아래 줄의 주석을 해제하세요.
# result = query(sql)
# display(result)

### 월별 매출 추이

In [ ]:
sql = """
-- AI가 만든 SELECT 쿼리를 여기에 넣으세요.
"""
# 쿼리를 넣은 뒤 아래 줄의 주석을 해제하세요.
# result = query(sql)
# display(result)

## 5. pandas 대조 예시 — 지역별 집계
먼저 실제 컬럼을 확인하세요. 아래 예시는 orders.order_total을 매출로 정의합니다. NULL 지역도 포함하고, 고객 키의 유일성과 주문 연결 누락을 확인합니다. SQL도 같은 기준으로 작성한 뒤 지역별로 숫자를 대조하세요.

In [ ]:
joined = orders.merge(customers[["customer_id", "region"]],
                      on="customer_id", how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(orders)
print("고객 연결 누락 주문:", int((joined["_merge"] == "left_only").sum()))
pandas_region = joined.groupby("region", dropna=False).agg(
    total_sales=("order_total", lambda s: s.sum(min_count=1)),
    order_count=("order_id", "size"),
).reset_index()
display(pandas_region)
print("원본 매출 합:", orders["order_total"].sum(min_count=1))
print("지역별 매출 합:", pandas_region["total_sales"].sum(min_count=1))

## 6. 제출 체크
- [ ] 집계 SELECT 3개
- [ ] SQL과 pandas의 지역·카테고리·월별 결과 대조표
- [ ] JOIN 전후 행 수와 원본 합계 확인
- [ ] NULL·중복·연결 누락 확인
- [ ] AI가 놓친 것과 수정 이유 한 줄

여기에 대조 결과와 수정 이유를 적으세요.

In [ ]:
conn.close()